# Customer Churn Prediction Using Machine Learning
**Developed by Swapna V**  
*M.Sc. Mathematics | AI & Machine Learning / Data Analytics*

## 1. Project Overview
This notebook provides a thorough walkthrough of exploratory data analysis, data cleaning, feature engineering, and cross-model evaluation for predicting customer churn in telecommunications.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

%matplotlib inline
sns.set_style('whitegrid')

## 2. Load & Clean Dataset

In [ ]:
df = pd.read_csv('../data/customer_churn.csv')
print(f"Initial shape: {df.shape}")

# Clean TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)

# Drop customer ID
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# Target encoding
if df['Churn'].dtype == object:
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df.info()

## 3. Exploratory Data Analysis (EDA)
We investigate key customer segments, contract types, monthly charges, and tenure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Churn Distribution
sns.countplot(data=df, x='Churn', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Overall Churn Distribution')

# Contract Type
sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[0, 1], palette='coolwarm')
axes[0, 1].set_title('Churn by Contract Type')

# Monthly Charges vs Churn
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[1, 0], palette='Set3')
axes[1, 0].set_title('Monthly Charges by Churn')

# Tenure vs Churn
sns.kdeplot(data=df, x='tenure', hue='Churn', common_norm=False, fill=True, ax=axes[1, 1], palette='tab10')
axes[1, 1].set_title('Tenure Distribution by Churn Status')

plt.tight_layout()
plt.show()

## 4. Pipeline & Model Training
Constructing an automated, leak-free preprocessing pipeline.

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42))
])

rf_pipe.fit(X_train, y_train)
preds = rf_pipe.predict(X_test)
print(classification_report(y_test, preds))